In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import re
import os
import warnings
import json

# NFI

In [18]:
# Cell 2 - Load NFI tables
stub = pd.read_csv('NFI/gunshot-residue-main/data/stub.csv')
source = pd.read_csv('NFI/gunshot-residue-main/data/source.csv')
stub_source = pd.read_csv('NFI/gunshot-residue-main/data/stub_source.csv')

print(f"Stubs:     {stub.shape}")
print(f"Sources:   {source.shape}")

Stubs:     (5029, 7)
Sources:   (1301, 4)


In [19]:
particle_files = sorted(glob.glob('NFI/gunshot-residue-main/data/particle_*.csv'))

#column names from file 1
cols = pd.read_csv(particle_files[0], nrows=0).columns.tolist()
print(f"Columns from particle_1: {len(cols)}")
dfs = [pd.read_csv(particle_files[0])]

# files 2-14 with no header, using file 1's column names
for f in particle_files[1:]:
    df = pd.read_csv(f, header=None, names=cols)
    dfs.append(df)

particle = pd.concat(dfs, ignore_index=True)
print(f"Particle shape: {particle.shape}")
print(f"Columns: {particle.columns.tolist()}")
print(particle.head())

Columns from particle_1: 93
Particle shape: (2801667, 93)
Columns: ['stub_id', 'particle_id', 'relevance_class', 'ac', 'ag', 'al', 'ar', 'as', 'at', 'au', 'b', 'ba', 'bi', 'br', 'ca', 'cd', 'ce', 'cl', 'co', 'cr', 'cs', 'cu', 'dy', 'er', 'eu', 'f', 'fe', 'fr', 'ga', 'gd', 'ge', 'hf', 'hg', 'ho', 'i', 'in', 'ir', 'k', 'kr', 'la', 'lu', 'mg', 'mn', 'mo', 'n', 'na', 'nb', 'nd', 'ne', 'ni', 'np', 'o', 'os', 'p', 'pa', 'pb', 'pd', 'pm', 'po', 'pr', 'pt', 'pu', 'ra', 'rb', 're', 'rh', 'rn', 'ru', 's', 'sb', 'sc', 'se', 'si', 'sm', 'sn', 'sr', 'ta', 'tb', 'tc', 'te', 'th', 'ti', 'tl', 'tm', 'u', 'v', 'w', 'xe', 'y', 'yb', 'zn', 'zr', 'merged_relevance_class']
   stub_id  particle_id relevance_class   ac   ag        al   ar   as   at  \
0       22         1454            PbSb  0.0  0.0  0.000000  0.0  0.0  0.0   
1       22         1274          PbSbBa  0.0  0.0  0.000000  0.0  0.0  0.0   
2       22          275          PbSbBa  0.0  0.0  0.751013  0.0  0.0  0.0   
3       22          714    

In [20]:
merge_map = {
    'PbSbBa': 'PbBaSb', 'PbSbBaSn': 'PbBaSb', 'PbSbBaSr': 'PbBaSb',
    'PbBa': 'PbBa', 'PbBaSn': 'PbBa',
    'PbSb': 'PbSb', 'PbSbSn': 'PbSb',
    'BaSb': 'BaSb', 'BaSbSn': 'BaSb',
    'Sb': 'Sb', 'SbSn': 'Sb',
    'Ba': 'Ba', 'BaSi': 'Ba', 'BaSr': 'Ba', 'BaSn': 'Ba',
    'BaAl': 'BaAl', 'BaAlS': 'BaAl',
    'BaCaSi': 'BaCaSi', 'BaCaSiS': 'BaCaSi',
    'CuZn': 'CuZn', 'Pb': 'Pb', 'ZnTi': 'ZnTi', 'Sr': 'Sr',
    'Hg': 'Hg', 'SbHg': 'Hg',
    'TiZnGd': 'TiZnGd', 'GaCuSn': 'GaCuSn',
}

particle['final_class'] = particle['relevance_class'].map(merge_map)
unmapped = particle[particle['final_class'].isna()]['relevance_class'].unique()
print(f"Unmapped classes: {unmapped}")
print(f"\nFinal class distribution:")
print(particle['final_class'].value_counts())

Unmapped classes: []

Final class distribution:
final_class
PbBaSb    536634
CuZn      426015
BaCaSi    375875
BaAl      341756
PbBa      236299
Sb        225676
PbSb      196294
Pb        168334
BaSb      109719
Ba         89277
ZnTi       44530
Sr         23395
TiZnGd     21479
Hg          4750
GaCuSn      1634
Name: count, dtype: int64


In [21]:
element_cols = [c for c in particle.columns if c not in [
    'stub_id', 'particle_id', 'relevance_class',
    'merged_relevance_class', 'final_class'
] and particle[c].dtype in ['float64', 'int64']]

print(f"Element columns ({len(element_cols)}):")
print("\n",element_cols)
print(f"\nSample of elemental data:")
print("\n",particle[element_cols].describe().round(3))

Element columns (89):

 ['ac', 'ag', 'al', 'ar', 'as', 'at', 'au', 'b', 'ba', 'bi', 'br', 'ca', 'cd', 'ce', 'cl', 'co', 'cr', 'cs', 'cu', 'dy', 'er', 'eu', 'f', 'fe', 'fr', 'ga', 'gd', 'ge', 'hf', 'hg', 'ho', 'i', 'in', 'ir', 'k', 'kr', 'la', 'lu', 'mg', 'mn', 'mo', 'n', 'na', 'nb', 'nd', 'ne', 'ni', 'np', 'o', 'os', 'p', 'pa', 'pb', 'pd', 'pm', 'po', 'pr', 'pt', 'pu', 'ra', 'rb', 're', 'rh', 'rn', 'ru', 's', 'sb', 'sc', 'se', 'si', 'sm', 'sn', 'sr', 'ta', 'tb', 'tc', 'te', 'th', 'ti', 'tl', 'tm', 'u', 'v', 'w', 'xe', 'y', 'yb', 'zn', 'zr']

Sample of elemental data:

                 ac           ag           al           ar           as  \
count  2801667.000  2801667.000  2801667.000  2801667.000  2801667.000   
mean         0.000        0.030        2.618        0.001        0.115   
std          0.023        1.177        8.063        0.058        0.825   
min          0.000        0.000        0.000        0.000        0.000   
25%          0.000        0.000        0.000        0.

In [23]:
particle_stub = particle.merge(
    stub, left_on='stub_id', right_on='id', suffixes=('_particle', '_stub')
)
print(f"Merged shape: {particle_stub.shape}")
print(f"\nSample types: {particle_stub['sample_type'].value_counts().to_dict()}")
print(f"Project types: {particle_stub['type_project'].value_counts().to_dict()}")

Merged shape: (2801667, 101)

Sample types: {'hand': 786171, 'cartridge': 570300, 'clothing': 325357, 'inshot': 104850, 'car': 75810, 'blanco': 11753, 'body': 9694, 'positive control': 3974}
Project types: {'zaak': 2432402, 'R&D': 369265}


### Note: zaak means case in Dutch.

# NIST

In [9]:
def parse_nist_sample(sample_path, source_label):
    """Extract particle classifications and class definitions from a NIST sample.
    Samples are stored in HDZ files."""

    hdz_file = None
    for candidate in ['mllsq.hdz', 'data.hdz']:
        
        if os.path.exists(os.path.join(sample_path, candidate)):
            hdz_file = os.path.join(sample_path, candidate)
            break
        for subdir in os.listdir(sample_path):
            sub_path = os.path.join(sample_path, subdir, candidate)
            if os.path.exists(sub_path):
                hdz_file = sub_path
                sample_path = os.path.join(sample_path, subdir)
                break
        if hdz_file:
            break

    if hdz_file is None:
        raise FileNotFoundError(f"No HDZ file found in {sample_path}")

    
    classes = {}
    elements = []
    with open(hdz_file, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.strip()
            match = re.match(r'CLASS(\d+)=(.*)', line)
            if match:
                classes[int(match.group(1))] = match.group(2)
            match = re.match(r'ELEM(\d+)=(\w+)', line)
            if match:
                elements.append(match.group(2))

    
    csv_file = None
    for candidate in ['MLLSQ_maxParticle.csv', 'maxPart.csv']:
        path_candidate = os.path.join(sample_path, candidate)
        if os.path.exists(path_candidate):
            csv_file = path_candidate
            break

    if csv_file is None:
        raise FileNotFoundError(f"No particle CSV found in {sample_path}")

    
    particles_df = pd.read_csv(csv_file, sep='\t', header=None,
                                names=['particle_start', 'particle_end', 'classification'])

    def extract_class(s):
        match = re.search(r'\[(.+?):', str(s))
        return match.group(1) if match else 'Unknown'

    particles_df['class_name'] = particles_df['classification'].apply(extract_class)

    
    rows = []
    for _, row in particles_df.iterrows():
        for pid in range(int(row['particle_start']), int(row['particle_end']) + 1):
            rows.append({
                'particle_id': pid,
                'class_name': row['class_name'],
                'source': source_label
            })

    result = pd.DataFrame(rows)


    def label_gsr(cls):
        cls_lower = cls.lower()
        if 'gsr' in cls_lower:
            return 'GSR'
        elif cls_lower in ['lowcounts', 'low counts', '#unclassified#']:
            return 'Low_Quality'
        else:
            return 'Non_GSR'

    result['gsr_label'] = result['class_name'].apply(label_gsr)

    return result, classes, elements

In [24]:
nist_samples = {
    'shooter1':     ('NIST/shooter/shooter1/',              'shooter_GSR'),
    'shooter2':     ('NIST/shooter/shooter2/',              'shooter_GSR'),
#    'sparklers':    ('NIST/firework/sparklers/',            'fireworks'),
#    'roman_candles':('NIST/firework/roman_candles/',        'fireworks'),
#    'spinners':     ('NIST/firework/spinners/',             'fireworks'),
#    'ford_explorer':('NIST/brakedust/ford_explorer/',       'brake_dust'),
}
nist_results = {}
nist_all_particles = []

for name, (dir_path, label) in nist_samples.items():
    print(f"\n{'='*50}")
    print(f"Processing: {name}")
    try:
        particles, classes, elements = parse_nist_sample(dir_path, label)
        nist_results[name] = {
            'particles': particles,
            'classes': classes,
            'elements': elements
        }
        nist_all_particles.append(particles)

        print(f"Particles: {len(particles)}")
        print(f"Elements measured: {elements}")
        print(f"\nGSR distribution:")
        print(f"{particles['gsr_label'].value_counts().to_dict()}")
        print(f"\nTop 10 classes:")
        print(particles['class_name'].value_counts().head(10).to_string())
    except Exception as e:
        print(f"Errored with: {e}")
        print(f"Contents of {dir_path}:")
        if os.path.exists(dir_path):
            for item in os.listdir(dir_path):
                print(f"    {item}")


Processing: shooter1
Particles: 3462
Elements measured: ['Na', 'Mg', 'Fe', 'Ni', 'Cu', 'Zn', 'Sr', 'Y', 'Ag', 'Sn', 'Sb', 'Ba', 'Al', 'Pt', 'Au', 'Pb', 'Bi', 'Si', 'S', 'Cl', 'K', 'Ca', 'Ti', 'Cr']

GSR distribution:
{'Non_GSR': 2313, 'GSR': 1014, 'Low_Quality': 135}

Top 10 classes:
class_name
Other             756
GSR.0             307
GSR.2             295
Ca-bearing        254
GSR.1             217
Lead              162
LowCounts         135
GSR.Sr-bearing    126
Nickel             99
Iron oxide         96

Processing: shooter2
Particles: 3429
Elements measured: ['Na', 'Mg', 'Cu', 'Zn', 'Ge', 'Mo', 'Sb', 'Ba', 'Pb', 'Bi', 'U', 'Al', 'Si', 'S', 'Ca', 'Ti', 'Cr', 'Fe', 'Ni']

GSR distribution:
{'Non_GSR': 1793, 'Low_Quality': 1524, 'GSR': 112}

Top 10 classes:
class_name
#Unclassified#     1524
Ca-bearing          436
Iron-bearing        265
Lead                239
Uranium-bearing     194
Char GSR-like       112
Iron                111
Silicate            103
Pb-Sb                75

In [25]:
nist_combined = pd.concat(nist_all_particles, ignore_index=True)
print(f"Total NIST particles: {len(nist_combined)}")

Total NIST particles: 6891


In [26]:
print("NIST: GSR labels by source:")
print(pd.crosstab(nist_combined['source'], nist_combined['gsr_label'], margins=True))

print("\nNIST: Detailed class distribution by source:")
ct = pd.crosstab(nist_combined['class_name'], nist_combined['source'])
ct = ct[ct.sum(axis=1) > 5]
print(ct.to_string())

NIST: GSR labels by source:
gsr_label     GSR  Low_Quality  Non_GSR   All
source                                       
shooter_GSR  1126         1659     4106  6891
All          1126         1659     4106  6891

NIST: Detailed class distribution by source:
source           shooter_GSR
class_name                  
#Unclassified#          1524
Aluminosilicate           21
Aluminum                  49
Au-Cu alloy               10
Ba-Sb                     41
Barite                    17
Bismuth                   80
Brass                     94
Ca-bearing               690
Calcite                   87
Char GSR-like            112
Chlorine-rich             22
Cr-bearing                66
Cu-rich                   79
Dolomite                   6
GSR.0                    307
GSR.1                    217
GSR.2                    295
GSR.Pb-Ba                 15
GSR.Pb-Sb                 52
GSR.Sr-bearing           126
Garnet                     6
Gold - other              75
Iron             

# Get labels for NFI data from NIST data

In [27]:
nfi_label_map = {
    # CONFIRMED GSR by NIST (found on known shooter samples)
    'PbBaSb':  'GSR',          # NIST: Classic GSR, GSR.0, GSR.1, GSR.2
    'PbBa':    'GSR',          # NIST: GSR.Pb-Ba
    'PbSb':    'GSR',          # NIST: GSR.Pb-Sb
    'BaSb':    'GSR',          # NIST: GSR.Ba-Sb

    # CONFIRMED NON-GSR by NIST (environmental/occupational)
    'BaAl':    'Non_GSR',      # No NIST GSR equivalent
    'BaCaSi':  'Non_GSR',      # No NIST GSR equivalent
    'CuZn':    'Non_GSR',      # NIST: Brass — found on shooters but NOT classified as GSR
    'ZnTi':    'Non_GSR',      # Environmental
    'Hg':      'Non_GSR',      # Environmental
    'TiZnGd':  'Non_GSR',      # Environmental
    'GaCuSn':  'Non_GSR',      # Environmental

    # AMBIGUOUS if single-element particles
    'Pb':      'Ambiguous',    # NIST: Lead class exists separate from GSR
    'Ba':      'Ambiguous',    # NIST: Barite (BaSO4) is environmental
    'Sb':      'Ambiguous',    # Could be GSR fragment or industrial
    'Sr':      'Ambiguous',    # NIST: GSR.Sr-bearing exists but also Celestine mineral
}

particle['label'] = particle['final_class'].map(nfi_label_map)

print("NFI Label Distribution:")
print(particle['label'].value_counts())
print(f"\nCross-tabulation")
print(pd.crosstab(particle['final_class'], particle['label'], margins=True))

NFI Label Distribution:
label
Non_GSR      1216039
GSR          1078946
Ambiguous     506682
Name: count, dtype: int64

Cross-tabulation
label        Ambiguous      GSR  Non_GSR      All
final_class                                      
Ba               89277        0        0    89277
BaAl                 0        0   341756   341756
BaCaSi               0        0   375875   375875
BaSb                 0   109719        0   109719
CuZn                 0        0   426015   426015
GaCuSn               0        0     1634     1634
Hg                   0        0     4750     4750
Pb              168334        0        0   168334
PbBa                 0   236299        0   236299
PbBaSb               0   536634        0   536634
PbSb                 0   196294        0   196294
Sb              225676        0        0   225676
Sr               23395        0        0    23395
TiZnGd               0        0    21479    21479
ZnTi                 0        0    44530    44530
All          

# For primary ML: use binary GSR vs Non_GSR + drop ambiguous

In [28]:
particle_binary = particle[particle['label'].isin(['GSR', 'Non_GSR'])].copy()
particle_binary['target'] = (particle_binary['label'] == 'GSR').astype(int)

print(f"Particles for binary classification: {len(particle_binary)}")
print(f"GSR: {particle_binary['target'].sum()}")
print(f"Non_GSR: {(1 - particle_binary['target']).sum()}")
print(f"Balance: {particle_binary['target'].mean():.1%} GSR")

Particles for binary classification: 2294985
GSR: 1078946
Non_GSR: 1216039
Balance: 47.0% GSR


In [29]:
for col in particle_binary.columns:
    print(col)

stub_id
particle_id
relevance_class
ac
ag
al
ar
as
at
au
b
ba
bi
br
ca
cd
ce
cl
co
cr
cs
cu
dy
er
eu
f
fe
fr
ga
gd
ge
hf
hg
ho
i
in
ir
k
kr
la
lu
mg
mn
mo
n
na
nb
nd
ne
ni
np
o
os
p
pa
pb
pd
pm
po
pr
pt
pu
ra
rb
re
rh
rn
ru
s
sb
sc
se
si
sm
sn
sr
ta
tb
tc
te
th
ti
tl
tm
u
v
w
xe
y
yb
zn
zr
merged_relevance_class
final_class
label
target


# Save for downstream tasks

In [30]:
particle.to_csv('processed_data/particle.csv', index=False) #main data

In [32]:
particle.to_parquet('processed_data/particle_labeled.parquet')